In [28]:
from dotenv import load_dotenv
import os
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-qe-eR7H_eASs


<img src='https://storage.googleapis.com/gweb-research2023-media/images/SufficientContext2_RAG.width-1250.png' />

In [29]:
chat = ChatOpenAI(temperature=0.1)

In [55]:
base_path = r"D:\back_0900_kdh\python\resource"

In [56]:
file_name = r"chapter_one.txt"

In [42]:
file_path = os.path.join(base_path, file_name)

In [43]:
print(file_path)

D:\back_0900_kdh\python\resource\chapter_one.txt


# 1. Text Loader

In [44]:
from langchain_community.document_loaders.text import TextLoader

In [45]:
loader = TextLoader(file_path)

In [46]:
docs = loader.load()

In [47]:
# List[Document]
docs

[Document(metadata={'source': 'D:\\back_0900_kdh\\python\\resource\\chapter_one.txt'}, page_content="Part 1, Chapter 1\n\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hat

# PDF Loader

In [60]:
from langchain_community.document_loaders.pdf import PyPDFLoader

In [61]:
loader = PyPDFLoader(file_path)

In [62]:
docs = loader.load()

invalid pdf header: b'PK\x03\x04\x14'
EOF marker not found


PdfStreamError: Stream has ended unexpectedly

# UnstructuredFileLoader

In [51]:
from langchain_community.document_loaders import UnstructuredFileLoader

In [53]:
file_name = r"chapter_one.docx"
file_path = os.path.join(base_path, file_name)

In [54]:
loader = UnstructuredFileLoader(file_path)

C:\Users\5418c\AppData\Local\Temp\ipykernel_33228\1033047943.py:1: LangChainDeprecationWarning: The class `UnstructuredFileLoader` was deprecated in LangChain 0.2.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-unstructured package and should be used instead. To use it run `pip install -U `langchain-unstructured` and import as `from `langchain_unstructured import UnstructuredLoader``.
  loader = UnstructuredFileLoader(file_path)


In [63]:
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

In [64]:
splitter = RecursiveCharacterTextSplitter()

In [65]:
# 들고 온 문서를 splitter에게 전달하는 방법
# \n\n
documents = splitter.split_documents(docs)

## chunk_size=
- 모델의 context window가 크지 않은 경우 좀 더 작은 document로 분리해야하며, chunk_size의 값으로 크기를 조절할 수 있다.

In [66]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200
)

In [67]:
# 문서를 들고올 때 잘라서 들고오기
documents = loader.load_and_split(docs, splitter=splitter)

TypeError: BaseLoader.load_and_split() got an unexpected keyword argument 'splitter'

In [68]:
print(len(documents))

11


In [69]:
print(documents[1].page_content)

Winston kept his back turned to the telescreen. It was safer, though, as he well knew, even a back can be revealing. A kilometre away the Ministry of Truth, his place of work, towered vast and white above the grimy landscape. This, he thought with a sort of vague distaste -- this was London, chief city of Airstrip One, itself the third most populous of the provinces of Oceania. He tried to squeeze out some childhood memory that should tell him whether London had always been quite like this. Were there always these vistas of rotting nineteenth-century houses, their sides shored up with baulks of timber, their windows patched with cardboard and their roofs with corrugated iron, their crazy garden walls sagging in all directions? And the bombed sites where the plaster dust swirled in the air and the willow-herb straggled over the heaps of rubble; and the places where the bombs had cleared a larger patch and there had sprung up sordid colonies of wooden dwellings like chicken-houses? But i

## chunk_overlap=
- 위와 같은 결과는 문장의 의미를 파괴할 수 있다. 작은 덩어리이면서 문장의 의미를 파괴하지 않는 방법이 chunk_overlap방법이다.
- 앞 조각의 끝 부분을 조금 가져와서 다음 조각에게 연결시켜 겹치는 부분(오버레이)가 생길 수 있지만, 문맥이 자연스럽게 연결된다.